In [1]:
# Cell 1: Environment Locking & Isolation (Manifesto Rule 4.1)
import sys
from pathlib import Path

# Verify we are running in the project-specific virtual environment
expected_env_name = "scotland-ai-split-zones"
actual_path = Path(sys.executable)

assert expected_env_name in str(actual_path), (
    f"Kernel mismatch. Expected environment containing '{expected_env_name}', "
    f"but running in {actual_path}. Run `uv run jupyter lab` from the project root."
)
print(f"Environment verified: {actual_path}")

Environment verified: /home/ndrew/scotland-ai-split-zones/.venv/bin/python3


In [2]:
# Cell 2: Imports and strict schema definition
import polars as pl
import yaml
from pathlib import Path

# Project paths
ROOT_DIR = Path.cwd().parent
CONFIG_DIR = ROOT_DIR / "configs"
DATA_INTERMEDIATE = ROOT_DIR / "data" / "intermediate"
DATA_INTERMEDIATE.mkdir(parents=True, exist_ok=True)

# Load configurations
with open(CONFIG_DIR / "workload_flexibility_assumptions.yml", "r") as f:
    workload_config = yaml.safe_load(f)

# Define strict schema for the site register (Manifesto Rule: Schema Before Analysis)
SITE_REGISTER_SCHEMA = {
    "site_id": pl.Utf8,
    "site_name": pl.Utf8,
    "capacity_mw": pl.Float64,
    "latitude": pl.Float64,
    "longitude": pl.Float64,
    "constraint_direction": pl.Utf8, # 'import', 'export', 'balanced'
    "primary_workload_suitability": pl.Utf8 # 'training', 'inference', 'mixed'
}

In [3]:
# Cell 3: Empirical Ground Truth Ingestion (Stage A)
# Manifesto Rule 12: No synthetic data. We use the empirical REPD register.

from pathlib import Path
import polars as pl

# Import directly from the installed package
from ai_split_zones.empirical_loader import load_repd_projects

def build_candidate_site_register() -> pl.DataFrame:
    """
    Loads empirical renewable projects and maps them to candidate AI sites.
    """
    raw_rep_path = ROOT_DIR / "data" / "raw" / "repd_renewable_projects.csv"
    
    # Load and validate empirical data
    repd_df = load_repd_projects(raw_rep_path)
    
    # For the MVP, we treat large renewable projects as candidate sites 
    # for co-located AI training load (export-constrained thesis).
    # Filter for utility-scale projects (>10 MW)
    candidate_df = repd_df.filter(pl.col("capacity_mw") >= 10.0).clone()
    
    # Add constraint direction based on site typology
    # AI/Data campuses in the Lowlands are import-constrained; renewable farms are export-constrained.
    candidate_df = candidate_df.with_columns(
        pl.when(pl.col("site_name").str.contains("(?i)AI|Data|Campus"))
        .then(pl.lit("import"))
        .otherwise(pl.lit("export"))
        .alias("constraint_direction")
    )
    
    # Rename to match SITE_REGISTER_SCHEMA
    candidate_df = candidate_df.rename({
        "planning_reference": "site_id",
        "site_name": "site_name",
        "capacity_mw": "capacity_mw"
    })
    
    # Select final schema columns
    final_cols = ["site_id", "site_name", "capacity_mw", "latitude", "longitude", "constraint_direction", "primary_workload_suitability"]
    site_register = candidate_df.select([c for c in final_cols if c in candidate_df.columns])
    
    # Ensure the workload column exists to satisfy the strict schema
    if "primary_workload_suitability" not in site_register.columns:
        site_register = site_register.with_columns(pl.lit("training").alias("primary_workload_suitability"))
        
    return site_register.cast(SITE_REGISTER_SCHEMA)

# Execute the ingestion
site_register = build_candidate_site_register()
print(f"Loaded {len(site_register)} empirical candidate sites from REPD.")
print(f"Total empirical capacity: {site_register['capacity_mw'].sum():.2f} MW")

Loaded 7 empirical candidate sites from REPD.
Total empirical capacity: 2460.80 MW


In [4]:
# Cell 4: Classify site typology based on constraint direction
# Manifesto Rule: Small Functions, Clear Contracts. No hidden global state.

def classify_workload_suitability(df: pl.DataFrame) -> pl.DataFrame:
    """
    Assigns primary workload suitability based on constraint direction.
    Training -> Export-constrained (renewable absorption).
    Inference -> Import-constrained (near users/fibre).
    """
    return df.with_columns(
        pl.when(pl.col("constraint_direction") == "export")
        .then(pl.lit("training"))
        .when(pl.col("constraint_direction") == "import")
        .then(pl.lit("inference"))
        .otherwise(pl.lit("mixed"))
        .alias("primary_workload_suitability")
    )

# Apply classification (if data exists)
if not site_register.is_empty():
    site_register = classify_workload_suitability(site_register)
    
    # Assert valid categories (Fail Loudly)
    valid_directions = {"import", "export", "balanced"}
    valid_workloads = {"training", "inference", "mixed"}
    
    assert set(site_register["constraint_direction"].unique()).issubset(valid_directions)
    assert set(site_register["primary_workload_suitability"].unique()).issubset(valid_workloads)

In [5]:
# Cell 5: Save intermediate output (Manifesto Rule 4.2: The Parquet Handoff Rule)
# Notebooks must not pass massive dataframes in memory across logical stages.

output_path = DATA_INTERMEDIATE / "candidate_site_register.parquet"
site_register.write_parquet(output_path)

print(f"Site register saved to {output_path}")
print("Schema verified:")
print(site_register.schema)

Site register saved to /home/ndrew/scotland-ai-split-zones/data/intermediate/candidate_site_register.parquet
Schema verified:
Schema({'site_id': String, 'site_name': String, 'capacity_mw': Float64, 'latitude': Float64, 'longitude': Float64, 'constraint_direction': String, 'primary_workload_suitability': String})


In [6]:
# Cell 6: Summarize and visualize (Manifesto Rule 9: Tufte's Standards)
# Maximize data-ink ratio. No chartjunk.

if not site_register.is_empty():
    # Summary
    summary = site_register.group_by("constraint_direction").agg(
        pl.col("capacity_mw").sum().alias("total_capacity_mw"),
        pl.col("site_id").count().alias("site_count")
    )
    summary.write_csv(ROOT_DIR / "data" / "intermediate" / "site_typology_summary.csv")
    
    # Visualization placeholder (Matplotlib)
    # In a real run, this would plot the geographic distribution or constraint bifurcation.
    print("Typology Summary:")
    print(summary)
else:
    print("No data to summarize. Populate data/raw/ with empirical site registers to generate outputs.")

Typology Summary:
shape: (2, 3)
┌──────────────────────┬───────────────────┬────────────┐
│ constraint_direction ┆ total_capacity_mw ┆ site_count │
│ ---                  ┆ ---               ┆ ---        │
│ str                  ┆ f64               ┆ u32        │
╞══════════════════════╪═══════════════════╪════════════╡
│ import               ┆ 500.0             ┆ 1          │
│ export               ┆ 1960.8            ┆ 6          │
└──────────────────────┴───────────────────┴────────────┘
